# BIOMASS TomoSAR — Plotting notebook

**Fourier • Capon • MUSIC • GEDI**

This notebook is dedicated to visualization, with no TomoSAR reconstruction or \(k_z\) interpolation. It loads the saved reconstruction cubes, GEDI, and the BIOMASS geometry; applies a GEDI-based height filter to derived height maps; then plots eight figures/groups of figures.

**Important:** Every range cut, azimuth cut, and pixel-profile cell creates **two separate Matplotlib figures**: (1) the corrected XP SLC with a red cut line/cross and cyan GEDI tracks, and (2) the tomographic result(s) with GEDI reference. This follows the supplied range-cut example. All output is saved in unique timestamped folders under `outputs_dir()`.

Run **Cells 1–5** in order, then run whichever plotting cells you need. Compatible saved MUSIC and Fourier/Capon grids are required. The SLC stack is opened with `mmap_mode="r"` to limit RAM use.

## 1 — Load saved Fourier, Capon, and MUSIC cubes

Select explicit run IDs if needed. Otherwise, load the newest *complete* Fourier/Capon run and newest MUSIC run with identical azimuth, range, and height sample axes. All cubes are memory-mapped.

In [ ]:
import json
from pathlib import Path
import numpy as np

from biomass_tomosar.paths import processed_data_dir

TOMOSAR_RUN_ID = None  # e.g. "20261123_150000_123456_UTC"
MUSIC_RUN_ID = None    # e.g. "20261127_000000_123456_UTC"

processed_dir = processed_data_dir()


def available_ids(prefix, suffix):
    return {
        p.name[len(prefix):-len(suffix)]
        for p in processed_dir.glob(f"{prefix}*{suffix}")
    }


complete_tomosar = sorted(
    available_ids("fourier_cube_", ".npy")
    & available_ids("capon_cube_", ".npy")
    & available_ids("tomosar_axes_", ".npz")
)
if not complete_tomosar:
    raise FileNotFoundError("No complete Fourier/Capon run with saved axes was found.")

tomosar_run_id = TOMOSAR_RUN_ID or complete_tomosar[-1]
if tomosar_run_id not in complete_tomosar:
    raise FileNotFoundError(f"Fourier/Capon run is missing files: {tomosar_run_id}")

cubeF = np.load(processed_dir / f"fourier_cube_{tomosar_run_id}.npy", mmap_mode="r")
cubeC = np.load(processed_dir / f"capon_cube_{tomosar_run_id}.npy", mmap_mode="r")

with np.load(processed_dir / f"tomosar_axes_{tomosar_run_id}.npz") as f:
    azax = f["azimuth_pixel_centers"].astype(int)
    rgax = f["range_pixel_centers"].astype(int)
    zaxis = f["height_m"].copy()


def music_grid_matches(run_id):
    with np.load(processed_dir / f"music_axes_{run_id}.npz") as f:
        return (
            np.array_equal(f["azimuth_pixel_centers"], azax)
            and np.array_equal(f["range_pixel_centers"], rgax)
            and np.array_equal(f["height_m"], zaxis)
        )


complete_music = sorted(
    available_ids("music_cube_", ".npy")
    & available_ids("music_axes_", ".npz")
)
if MUSIC_RUN_ID is not None:
    if MUSIC_RUN_ID not in complete_music or not music_grid_matches(MUSIC_RUN_ID):
        raise ValueError("Requested MUSIC run is absent or has an incompatible grid.")
    music_run_id = MUSIC_RUN_ID
else:
    compatible_music = [rid for rid in complete_music if music_grid_matches(rid)]
    if not compatible_music:
        raise FileNotFoundError("No saved MUSIC cube has the same grid as Fourier/Capon.")
    music_run_id = compatible_music[-1]

cubeM = np.load(processed_dir / f"music_cube_{music_run_id}.npy", mmap_mode="r")

expected_shape = (len(azax), len(rgax), len(zaxis))
for method, cube in [("Fourier", cubeF), ("Capon", cubeC), ("MUSIC", cubeM)]:
    if cube.shape != expected_shape:
        raise ValueError(f"{method}: cube shape {cube.shape} != {expected_shape}")

music_meta_path = processed_dir / f"music_metadata_{music_run_id}.json"
music_metadata = {}
if music_meta_path.exists():
    with music_meta_path.open() as f:
        music_metadata = json.load(f)
N_SIGNALS = music_metadata.get("n_signal_sources")
MUSIC_LABEL = f"MUSIC (Ns={N_SIGNALS})" if N_SIGNALS is not None else "MUSIC"

print(f"Fourier/Capon run: {tomosar_run_id}")
print(f"MUSIC run:         {music_run_id}")
print(f"MUSIC N_SIGNALS:   {N_SIGNALS}")
print(f"All cubes:        {expected_shape}")
print(f"Height interval:  {zaxis[0]}–{zaxis[-1]} m")

## 2 — Load GEDI canopy heights and raw GEDI measurements

The gridded GEDI reference uses `gedi_forestHeight.npy` and its `.grd` metadata. The raw point measurements (`gedi.csv`) are retained for overlay on cuts, following your reference figure.

In [ ]:
import pandas as pd
from biomass_tomosar.paths import gedi_data_dir

gedi_dir = gedi_data_dir()

gedi_height = np.load(gedi_dir / "gedi_forestHeight.npy", mmap_mode="r")
grid = pd.read_csv(gedi_dir / "gedi_forestHeight.grd").iloc[0]

az0, naz, daz = float(grid["az0"]), int(grid["naz"]), float(grid["daz"])
rg0, nrg, drg = float(grid["rg0"]), int(grid["nrg"]), float(grid["drg"])

gedi_az = az0 + daz * np.arange(naz)
gedi_rg = rg0 + drg * np.arange(nrg)
assert gedi_height.shape == (naz, nrg)

gedi_points = pd.read_csv(
    gedi_dir / "gedi.csv",
    usecols=["rg", "az", "rh_98", "rh_0"],
)
gedi_points["forest_height"] = gedi_points["rh_98"] - gedi_points["rh_0"]
gedi_points = gedi_points.loc[
    np.isfinite(gedi_points["rg"])
    & np.isfinite(gedi_points["az"])
    & np.isfinite(gedi_points["forest_height"])
    & (gedi_points["forest_height"] >= 0)
].copy()

print("GEDI map:", gedi_height.shape)
print("Raw valid GEDI shots:", len(gedi_points))
print(f"GEDI range: {gedi_rg[0]/1000:.3f}–{gedi_rg[-1]/1000:.3f} km")

## 3 — Initialize `bmStack`, load corrected XP SLC, and restore SAR coordinates

Uses the acquisition order in `xp_stack_metadata.json` when present. `bmStack` reads BIOMASS annotations/LUT descriptions; the full-resolution corrected XP array is opened from `xp_stack.npy` by memory mapping, **without recomputing SKP or flattening corrections**.

In [ ]:
from biomass_tomosar.paths import biomass_stack_dir
from biomass_tomosar.bm_data_loader import bmStack

stack_dir = biomass_stack_dir()
slc_metadata_path = processed_dir / "xp_stack_metadata.json"

if slc_metadata_path.exists():
    with slc_metadata_path.open() as f:
        xp_metadata = json.load(f)
    products = [stack_dir / name for name in xp_metadata["acquisitions"]]
    MASTER_IDX = int(xp_metadata["reference_acquisition_index"])
else:
    products = sorted(p for p in stack_dir.glob("BIO*") if p.is_dir())
    MASTER_IDX = len(products) // 2
    print("Warning: XP acquisition metadata absent; using sorted BIOMASS product order.")

if not products or not all(p.is_dir() for p in products):
    raise FileNotFoundError("Missing BIOMASS acquisition product folder(s).")

stack = bmStack([str(p) for p in products])
xp_stack = np.load(processed_dir / "xp_stack.npy", mmap_mode="r")
n_acq, n_az, n_rg = xp_stack.shape

if n_acq != len(products):
    raise ValueError("Corrected XP stack and BIOMASS product counts differ.")
if not (0 <= MASTER_IDX < n_acq):
    raise ValueError("Invalid reference acquisition index.")

annot = stack.annot_coreg[MASTER_IDX]


def xml_float(name):
    nodes = annot.xpath(f"//*[local-name()='sarImage']/*[local-name()='{name}']")
    if not nodes or nodes[0].text is None:
        raise ValueError(f"Missing SAR annotation: {name}")
    return float(nodes[0].text)


C = 299_792_458.0
az_spacing = xml_float("azimuthPixelSpacing")
rg_spacing = xml_float("rangePixelSpacing")
first_range_time = xml_float("firstSampleSlantRangeTime")

sar_az = np.arange(n_az) * az_spacing
sar_rg = C * first_range_time / 2 + np.arange(n_rg) * rg_spacing

tomo_az = sar_az[azax]
tomo_rg = sar_rg[rgax]

assert cubeF.shape == cubeC.shape == cubeM.shape
assert cubeF.shape == (len(tomo_az), len(tomo_rg), len(zaxis))

print(f"BIOMASS acquisitions: {n_acq} | master index: {MASTER_IDX}")
print(f"Corrected XP SLC:     {xp_stack.shape} (memory-mapped)")
print(f"SAR pixel spacing:    {az_spacing:.3f} m (az), {rg_spacing:.3f} m (rg)")
print(f"TomoSAR coordinates:  {len(tomo_az)} az × {len(tomo_rg)} rg")

## 4 — Height filter derived from GEDI

Calculate `HEIGHT_THRESHOLD = max(valid GEDI height) + 5 m`. Derive the height of the **original maximum** of each TomoSAR vertical profile, and replace peak-height estimates **above** the threshold with `NaN`. The original `cubeF`, `cubeC`, and `cubeM` are never altered; range cuts, azimuth cuts, and pixel profiles therefore remain unfiltered.

In [ ]:
# =========================================================
# FILTER PARAMETERS
# =========================================================
HEIGHT_MARGIN_M = 5.0
APPLY_GEDI_HEIGHT_FILTER = True

gedi_valid_grid = np.ma.filled(np.ma.asarray(gedi_height, dtype=float), np.nan)
gedi_valid_values = gedi_valid_grid[
    np.isfinite(gedi_valid_grid) & (gedi_valid_grid >= 0)
]
if not gedi_valid_values.size:
    raise ValueError("No valid GEDI canopy heights available.")

GEDI_MAX_HEIGHT = float(gedi_valid_values.max())
HEIGHT_THRESHOLD = GEDI_MAX_HEIGHT + HEIGHT_MARGIN_M


def peak_height_map(cube, chunk_size=32):
    """Peak-scattering height [m], with optional global GEDI-derived upper mask."""
    result = np.full(cube.shape[:2], np.nan, dtype=np.float32)
    for i in range(0, cube.shape[0], chunk_size):
        j = min(i + chunk_size, cube.shape[0])
        p = np.asarray(cube[i:j])
        valid = np.isfinite(p) & (p > 0)
        indices = np.argmax(np.where(valid, p, -np.inf), axis=2)
        heights = np.asarray(zaxis[indices], dtype=np.float32)
        heights[~np.any(valid, axis=2)] = np.nan
        if APPLY_GEDI_HEIGHT_FILTER:
            heights[heights > HEIGHT_THRESHOLD] = np.nan
        result[i:j] = heights
    return result


fourier_height_map = peak_height_map(cubeF)
capon_height_map = peak_height_map(cubeC)
music_height_map = peak_height_map(cubeM)

print(f"GEDI max canopy height: {GEDI_MAX_HEIGHT:.2f} m")
print(f"Upper height threshold: {HEIGHT_THRESHOLD:.2f} m")
for name, hmap in [("Fourier", fourier_height_map), ("Capon", capon_height_map),
                   ("MUSIC", music_height_map)]:
    count = int(np.isfinite(hmap).sum())
    print(f"{name:8}: {count:,} / {hmap.size:,} heights retained")

## 5 — Shared plotting settings and helpers

**Reference range-cut style preserved:** separate 9×11 in SLC overview, GEDI tracks as cyan points, red line/cross marking the selected position, downward-increasing azimuth image display; separate tomographic figure with `pcolormesh`, cyan gridded GEDI canopy curve, and lime raw GEDI shots near the cut; default non-normalized dB for cut maps and individual color scales per method.

Edit coordinates in **kilometres**. `None` selects the center of the common TomoSAR/GEDI spatial overlap. `RANGE_CUT_KM = 768.0` reproduces the attached example.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from datetime import datetime, timezone
from biomass_tomosar.paths import outputs_dir

# =========================================================
# 1. SETTINGS
# =========================================================
ACQ_IDX = 0
SLC_DISPLAY_STEP = 6
GEDI_TRACK_DISPLAY_STEP = 10

RANGE_CUT_KM = 768.0    # As in supplied example; None = overlap center
AZIMUTH_CUT_KM = None   # None = overlap center
PIXEL_AZ_KM = None
PIXEL_RG_KM = None

NORMALIZE_CUT_PROFILES = False     # Reference range-cut default
NORMALIZE_PIXEL_PROFILES = True   # Better for comparing spectral shapes
DYNAMIC_RANGE_DB = 30

SAVE_FIGURES = True
FIGURE_DPI = 200

HEIGHT_MIN = 0
HEIGHT_MAX = HEIGHT_THRESHOLD  # GEDI max + 5 m
SURFACE_MAX_POINTS = 120
SURFACE_ELEV = 28
SURFACE_AZIM = -65
HEIGHT_CMAP = "viridis"
TOMOGRAM_CMAP = "magma"

if not (0 <= ACQ_IDX < n_acq):
    raise ValueError("ACQ_IDX is outside the SLC acquisition stack.")

# =========================================================
# 2. SAVE EACH PLOT CELL IN A NEW FOLDER
# =========================================================
def save_plot_set(folder_name, figures):
    """figures: dictionary mapping filename stems to Figure objects."""
    if not SAVE_FIGURES:
        return None
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
    run_dir = outputs_dir() / f"{folder_name}_{stamp}"
    run_dir.mkdir(parents=True, exist_ok=False)
    for filename_stem, fig in figures.items():
        path = run_dir / f"{filename_stem}.png"
        with path.open("xb") as f:
            fig.savefig(f, format="png", dpi=FIGURE_DPI, bbox_inches="tight")
        print("Saved:", path)
    return run_dir

# =========================================================
# 3. SPATIAL SELECTION (OVERLAP-AWARE)
# =========================================================
def choose_cut(tomo_axis_m, gedi_axis_m, requested_km, name):
    lo = max(float(min(tomo_axis_m)), float(min(gedi_axis_m)))
    hi = min(float(max(tomo_axis_m)), float(max(gedi_axis_m)))
    if lo >= hi:
        raise ValueError(f"No TomoSAR/GEDI overlap on {name} axis.")
    target = (lo + hi) / 2 if requested_km is None else float(requested_km) * 1000
    if not (lo <= target <= hi):
        raise ValueError(
            f"{name} = {target/1000:.3f} km outside common coverage "
            f"[{lo/1000:.3f}, {hi/1000:.3f}] km"
        )
    idx = int(np.argmin(np.abs(tomo_axis_m - target)))
    return idx, float(tomo_axis_m[idx])


def select_range_cut():
    return choose_cut(tomo_rg, gedi_rg, RANGE_CUT_KM, "Range")


def select_azimuth_cut():
    return choose_cut(tomo_az, gedi_az, AZIMUTH_CUT_KM, "Azimuth")


def select_pixel():
    ia, az_m = choose_cut(tomo_az, gedi_az, PIXEL_AZ_KM, "Pixel azimuth")
    ir, rg_m = choose_cut(tomo_rg, gedi_rg, PIXEL_RG_KM, "Pixel range")
    return ia, ir, az_m, rg_m

# =========================================================
# 4. SLC DISPLAY (SEPARATE FIGURE)
# =========================================================
slc_sample = np.asarray(
    xp_stack[ACQ_IDX, ::SLC_DISPLAY_STEP, ::SLC_DISPLAY_STEP]
)
slc_amplitude = np.abs(slc_sample)
slc_db = 20 * np.log10(np.maximum(slc_amplitude, 1e-8))
slc_good = slc_db[np.isfinite(slc_db) & (slc_amplitude > 0)]
slc_vmin, slc_vmax = (
    np.percentile(slc_good, [2, 98]) if slc_good.size else (-40.0, 0.0)
)
slc_extent = [
    sar_rg[0]/1000, sar_rg[-1]/1000,
    sar_az[0]/1000, sar_az[-1]/1000,
]


def make_slc_figure(kind, az_m=None, rg_m=None):
    """Independent SLC figure with GEDI tracks and selected cut/point."""
    fig, ax = plt.subplots(figsize=(9, 11), constrained_layout=True)
    im = ax.imshow(
        slc_db, extent=slc_extent, origin="lower", aspect="auto",
        cmap="gray", vmin=slc_vmin, vmax=slc_vmax,
    )
    if kind == "range":
        ax.axvline(rg_m/1000, color="red", linewidth=2,
                   label=f"Range cut: {rg_m/1000:.3f} km")
    elif kind == "azimuth":
        ax.axhline(az_m/1000, color="red", linewidth=2,
                   label=f"Azimuth cut: {az_m/1000:.3f} km")
    elif kind == "pixel":
        ax.plot(rg_m/1000, az_m/1000, marker="x", color="red",
                markersize=14, markeredgewidth=3, linestyle="none",
                label="Selected TomoSAR pixel", zorder=5)
    else:
        raise ValueError(f"Unknown SLC annotation: {kind}")

    pts = gedi_points.iloc[::GEDI_TRACK_DISPLAY_STEP]
    ax.scatter(
        pts["rg"]/1000, pts["az"]/1000,
        s=1, color="cyan", alpha=0.25, rasterized=True,
        label="GEDI tracks",
    )
    ax.set_title(f"Corrected XP SLC — Acquisition {ACQ_IDX + 1}")
    ax.set_xlabel("Slant range [km]")
    ax.set_ylabel("Azimuth [km]")
    ax.set_xlim(slc_extent[0], slc_extent[1])
    ax.set_ylim(slc_extent[3], slc_extent[2])
    ax.legend(loc="upper right")
    fig.colorbar(im, ax=ax, label="SLC amplitude [dB]", shrink=0.7)
    return fig

# =========================================================
# 5. TOMOGRAM CUT WITH BOTH GRIDDED AND RAW GEDI
# =========================================================
def prepare_spectrum_db(section, normalize=False):
    """Convert (space, height) section to (height, space) dB."""
    power = np.asarray(section, dtype=np.float64)
    valid = np.isfinite(power) & (power > 0)
    if normalize:
        peaks = np.max(np.where(valid, power, 0), axis=1, keepdims=True)
        out = np.full_like(power, np.nan)
        np.divide(power, peaks, out=out, where=valid & (peaks > 0))
        power = out
        valid = np.isfinite(power) & (power > 0)
    db = np.full(power.shape, np.nan, dtype=np.float64)
    db[valid] = 10 * np.log10(power[valid])
    return db.T


def color_limits(db, normalize):
    if normalize:
        return -DYNAMIC_RANGE_DB, 0.0
    good = db[np.isfinite(db)]
    vmax = float(np.percentile(good, 99)) if good.size else 0.0
    return vmax - DYNAMIC_RANGE_DB, vmax


def clean_gedi_heights(heights):
    h = np.ma.filled(np.ma.asarray(heights, dtype=float), np.nan).copy()
    h[~np.isfinite(h) | (h < 0)] = np.nan
    return h


def plot_tomogram(ax, cube, kind, idx, title, spectrum_label):
    """Range: height-vs-azimuth; azimuth: height-vs-range."""
    if kind == "range":
        section = cube[:, idx, :]
        x_m = tomo_az
        cut_m = tomo_rg[idx]
        gedi_idx = int(np.argmin(np.abs(gedi_rg - cut_m)))
        gedi_x = gedi_az
        gedi_h = clean_gedi_heights(gedi_height[:, gedi_idx])
        tomo_step = float(np.median(np.diff(tomo_rg)))
        half_width = max(30.0, tomo_step / 2)
        near_cut = gedi_points.loc[
            np.abs(gedi_points["rg"] - cut_m) <= half_width
        ]
        point_axis = "az"
        x_label = "Azimuth [km]"
    elif kind == "azimuth":
        section = cube[idx, :, :]
        x_m = tomo_rg
        cut_m = tomo_az[idx]
        gedi_idx = int(np.argmin(np.abs(gedi_az - cut_m)))
        gedi_x = gedi_rg
        gedi_h = clean_gedi_heights(gedi_height[gedi_idx, :])
        tomo_step = float(np.median(np.diff(tomo_az)))
        half_width = max(30.0, tomo_step / 2)
        near_cut = gedi_points.loc[
            np.abs(gedi_points["az"] - cut_m) <= half_width
        ]
        point_axis = "rg"
        x_label = "Slant range [km]"
    else:
        raise ValueError("kind must be 'range' or 'azimuth'")

    db = prepare_spectrum_db(section, normalize=NORMALIZE_CUT_PROFILES)
    vmin, vmax = color_limits(db, NORMALIZE_CUT_PROFILES)
    im = ax.pcolormesh(
        x_m/1000, zaxis, db, shading="nearest",
        cmap=TOMOGRAM_CMAP, vmin=vmin, vmax=vmax,
        rasterized=True,
    )
    ax.plot(
        gedi_x/1000, gedi_h, color="cyan", linewidth=1.8,
        label="GEDI canopy height", zorder=4,
    )
    if not near_cut.empty:
        ax.scatter(
            near_cut[point_axis]/1000,
            near_cut["forest_height"],
            s=8, color="lime", alpha=0.55,
            label="Raw GEDI", rasterized=True, zorder=5,
        )
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Height [m]")
    ax.set_xlim(x_m[0]/1000, x_m[-1]/1000)
    ax.set_ylim(zaxis[0], zaxis[-1])
    ax.grid(alpha=0.15)
    ax.legend(loc="upper right")
    return im, (
        "Relative spectrum [dB]" if NORMALIZE_CUT_PROFILES
        else spectrum_label
    )

# =========================================================
# 6. PIXEL PROFILES AND GEDI REFERENCE
# =========================================================
def pixel_gedi_height(az_m, rg_m):
    ga = int(np.argmin(np.abs(gedi_az - az_m)))
    gr = int(np.argmin(np.abs(gedi_rg - rg_m)))
    height = float(np.ma.filled(np.ma.asarray(gedi_height[ga, gr]), np.nan))
    return height if np.isfinite(height) and height >= 0 else None


def pixel_spectrum_db(cube, ia, ir):
    p = np.asarray(cube[ia, ir, :], dtype=np.float64)
    good = np.isfinite(p) & (p > 0)
    p = np.where(good, p, np.nan)
    if NORMALIZE_PIXEL_PROFILES and np.any(good):
        p = p / np.nanmax(p)
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(p)


def make_pixel_profile_figure(methods, ia, ir, az_m, rg_m):
    """methods is a list of (label, cube, color)."""
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    for label, cube, color in methods:
        db = pixel_spectrum_db(cube, ia, ir)
        ax.plot(zaxis, db, label=label, linewidth=2, color=color)

    gedi_h = pixel_gedi_height(az_m, rg_m)
    if gedi_h is not None:
        ax.axvline(
            gedi_h, color="green", linestyle="--", linewidth=2,
            label=f"GEDI canopy height: {gedi_h:.1f} m",
        )
    ax.set_title(f"Vertical profile — Az {az_m/1000:.3f} km, Rg {rg_m/1000:.3f} km")
    ax.set_xlabel("Height [m]")
    ax.set_ylabel(
        "Relative spectrum [dB]" if NORMALIZE_PIXEL_PROFILES
        else "Spectrum [dB] (method-dependent units)"
    )
    ax.set_xlim(zaxis[0], zaxis[-1])
    if NORMALIZE_PIXEL_PROFILES:
        ax.set_ylim(-DYNAMIC_RANGE_DB, 2)
    ax.grid(alpha=0.3)
    ax.legend(loc="best")
    return fig

# =========================================================
# 7. 3D PEAK HEIGHT SURFACES
# =========================================================
def plot_height_surface(ax, height_map, az_m, rg_m, title):
    step_a = max(1, int(np.ceil(len(az_m) / SURFACE_MAX_POINTS)))
    step_r = max(1, int(np.ceil(len(rg_m) / SURFACE_MAX_POINTS)))
    z = clean_gedi_heights(height_map[::step_a, ::step_r])
    z[(z < HEIGHT_MIN) | (z > HEIGHT_MAX)] = np.nan
    x, y = np.meshgrid(rg_m[::step_r]/1000, az_m[::step_a]/1000)
    ax.plot_surface(
        x, y, z, cmap=HEIGHT_CMAP,
        vmin=HEIGHT_MIN, vmax=HEIGHT_MAX,
        linewidth=0, antialiased=False,
        rcount=z.shape[0], ccount=z.shape[1],
    )
    ax.set_title(title, pad=12)
    ax.set_xlabel("Slant range [km]")
    ax.set_ylabel("Azimuth [km]")
    ax.set_zlabel("Height [m]")
    ax.set_xlim(tomo_rg[0]/1000, tomo_rg[-1]/1000)
    ax.set_ylim(tomo_az[-1]/1000, tomo_az[0]/1000)
    ax.set_zlim(HEIGHT_MIN, HEIGHT_MAX)
    ax.view_init(elev=SURFACE_ELEV, azim=SURFACE_AZIM)
    ax.set_box_aspect((1, 1.3, 0.65))


def add_height_colorbar(fig, axes):
    norm = mpl.colors.Normalize(HEIGHT_MIN, HEIGHT_MAX)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=HEIGHT_CMAP)
    sm.set_array([])
    fig.colorbar(sm, ax=axes, label="Height [m]", shrink=0.62, pad=0.08)

print("Plotting helpers ready.")
print(f"GEDI upper height threshold: {HEIGHT_THRESHOLD:.2f} m")
print("Run any plot cell below; SLC figures are always separate.")

---
# A — MUSIC only

Each cut/profile creates a **separate SLC figure**, followed by its MUSIC-only plot. The 3D figure compares MUSIC peak scattering height against GEDI canopy height.

## 6 — MUSIC range cut vs GEDI (+ separate SLC)

In [ ]:
RG_IDX, cut_rg_m = select_range_cut()
print(f"MUSIC range cut at {cut_rg_m/1000:.3f} km (index {RG_IDX})")

fig_slc = make_slc_figure("range", rg_m=cut_rg_m)

fig_tomo, ax = plt.subplots(figsize=(15, 6), constrained_layout=True)
im, label = plot_tomogram(
    ax, cubeM, "range", RG_IDX,
    f"{MUSIC_LABEL} — Range {cut_rg_m/1000:.3f} km",
    "MUSIC pseudo-spectrum [dB]",
)
fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)
fig_tomo.suptitle("MUSIC TomoSAR vs GEDI — range cut", fontsize=14)

save_plot_set("music_range_cut", {
    "slc_overview": fig_slc,
    "music_gedi_range_cut": fig_tomo,
})
plt.show()

## 7 — MUSIC azimuth cut vs GEDI (+ separate SLC)

In [ ]:
AZ_IDX, cut_az_m = select_azimuth_cut()
print(f"MUSIC azimuth cut at {cut_az_m/1000:.3f} km (index {AZ_IDX})")

fig_slc = make_slc_figure("azimuth", az_m=cut_az_m)

fig_tomo, ax = plt.subplots(figsize=(15, 6), constrained_layout=True)
im, label = plot_tomogram(
    ax, cubeM, "azimuth", AZ_IDX,
    f"{MUSIC_LABEL} — Azimuth {cut_az_m/1000:.3f} km",
    "MUSIC pseudo-spectrum [dB]",
)
fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)
fig_tomo.suptitle("MUSIC TomoSAR vs GEDI — azimuth cut", fontsize=14)

save_plot_set("music_azimuth_cut", {
    "slc_overview": fig_slc,
    "music_gedi_azimuth_cut": fig_tomo,
})
plt.show()

## 8 — MUSIC pixel profile vs GEDI (+ separate SLC)

In [ ]:
ia, ir, az_m, rg_m = select_pixel()
print(f"Selected pixel: az {az_m/1000:.3f} km, range {rg_m/1000:.3f} km")

fig_slc = make_slc_figure("pixel", az_m=az_m, rg_m=rg_m)
fig_profile = make_pixel_profile_figure(
    [(MUSIC_LABEL, cubeM, "tab:purple")], ia, ir, az_m, rg_m
)

profile = np.asarray(cubeM[ia, ir, :])
if np.any(np.isfinite(profile) & (profile > 0)):
    print(f"MUSIC peak: {zaxis[np.nanargmax(profile)]:.1f} m")
print("GEDI canopy height:", pixel_gedi_height(az_m, rg_m))

save_plot_set("music_pixel_profile", {
    "slc_overview": fig_slc,
    "music_gedi_pixel_profile": fig_profile,
})
plt.show()

## 9 — MUSIC vs GEDI 3D height surfaces

The MUSIC surface uses `music_height_map` from Cell 4, not the raw cube. The GEDI-derived upper height filter applies to the MUSIC **peak-height map**. No SLC overlay is needed for a height-map view.

In [ ]:
fig_3d = plt.figure(figsize=(16, 8), constrained_layout=True)
axes = [fig_3d.add_subplot(1, 2, i+1, projection="3d") for i in range(2)]

plot_height_surface(
    axes[0], music_height_map, tomo_az, tomo_rg,
    f"{MUSIC_LABEL} — Peak scattering height",
)
plot_height_surface(
    axes[1], gedi_height, gedi_az, gedi_rg,
    "GEDI — Canopy height",
)
add_height_colorbar(fig_3d, axes)
fig_3d.suptitle("MUSIC and GEDI — 3D height maps", fontsize=15)

save_plot_set("music_vs_gedi_3d", {"music_vs_gedi_3d": fig_3d})
plt.show()

---
# B — Fourier + Capon + MUSIC comparisons

For range and azimuth cuts, the **SLC overview is its own figure**. The second figure contains three vertically stacked tomograms with matching spatial coordinates, each with a separate colorbar and its own interpolated/raw GEDI overlays (as in the supplied reference cell). For the pixel cut, all three normalized profiles share one axis.

## 10 — Three-method range cut vs GEDI (+ separate SLC)

In [ ]:
RG_IDX, cut_rg_m = select_range_cut()
fig_slc = make_slc_figure("range", rg_m=cut_rg_m)

fig_tomo, axes = plt.subplots(
    3, 1, figsize=(15, 14), sharex=True, sharey=True,
    constrained_layout=True,
)
methods = [
    ("Fourier TomoSAR", cubeF, "Fourier power [dB]"),
    ("Capon TomoSAR", cubeC, "Capon power [dB]"),
    (f"{MUSIC_LABEL} TomoSAR", cubeM, "MUSIC pseudo-spectrum [dB]"),
]
for ax, (name, cube, unit) in zip(axes, methods):
    im, label = plot_tomogram(ax, cube, "range", RG_IDX, name, unit)
    fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)

mode = "Normalized" if NORMALIZE_CUT_PROFILES else "Unnormalized"
fig_tomo.suptitle(
    f"Fourier / Capon / MUSIC — {mode} spectra\n"
    f"Range: {cut_rg_m/1000:.3f} km",
    fontsize=15,
)

save_plot_set("comparison_range_cut", {
    "slc_overview": fig_slc,
    "fourier_capon_music_gedi_range_cut": fig_tomo,
})
plt.show()

## 11 — Three-method azimuth cut vs GEDI (+ separate SLC)

In [ ]:
AZ_IDX, cut_az_m = select_azimuth_cut()
fig_slc = make_slc_figure("azimuth", az_m=cut_az_m)

fig_tomo, axes = plt.subplots(
    3, 1, figsize=(15, 14), sharex=True, sharey=True,
    constrained_layout=True,
)
methods = [
    ("Fourier TomoSAR", cubeF, "Fourier power [dB]"),
    ("Capon TomoSAR", cubeC, "Capon power [dB]"),
    (f"{MUSIC_LABEL} TomoSAR", cubeM, "MUSIC pseudo-spectrum [dB]"),
]
for ax, (name, cube, unit) in zip(axes, methods):
    im, label = plot_tomogram(ax, cube, "azimuth", AZ_IDX, name, unit)
    fig_tomo.colorbar(im, ax=ax, label=label, shrink=0.85)

mode = "Normalized" if NORMALIZE_CUT_PROFILES else "Unnormalized"
fig_tomo.suptitle(
    f"Fourier / Capon / MUSIC — {mode} spectra\n"
    f"Azimuth: {cut_az_m/1000:.3f} km",
    fontsize=15,
)

save_plot_set("comparison_azimuth_cut", {
    "slc_overview": fig_slc,
    "fourier_capon_music_gedi_azimuth_cut": fig_tomo,
})
plt.show()

## 12 — Three-method pixel profile vs GEDI (+ separate SLC)

Default: each method's profile is normalized to its own maximum (0 dB), making peak positions and widths directly comparable despite MUSIC having a non-physical pseudo-spectrum amplitude.

In [ ]:
ia, ir, az_m, rg_m = select_pixel()
fig_slc = make_slc_figure("pixel", az_m=az_m, rg_m=rg_m)

methods = [
    ("Fourier", cubeF, "tab:blue"),
    ("Capon", cubeC, "tab:orange"),
    (MUSIC_LABEL, cubeM, "tab:purple"),
]
fig_profile = make_pixel_profile_figure(methods, ia, ir, az_m, rg_m)

for name, cube, _ in methods:
    prof = np.asarray(cube[ia, ir, :])
    valid = np.isfinite(prof) & (prof > 0)
    if np.any(valid):
        idx = np.argmax(np.where(valid, prof, -np.inf))
        print(f"{name:15} peak: {zaxis[idx]:.1f} m")
print("GEDI canopy height:", pixel_gedi_height(az_m, rg_m))

save_plot_set("comparison_pixel_profile", {
    "slc_overview": fig_slc,
    "fourier_capon_music_gedi_pixel_profile": fig_profile,
})
plt.show()

## 13 — Four 3D height surfaces in a 2×2 layout

**Top row:** Fourier and Capon. **Bottom row:** MUSIC and GEDI. All use the same height color scale (`0` to `max(GEDI)+5 m`). The three radar surfaces represent **dominant scattering height**, while GEDI represents canopy height above its ground reference.

In [ ]:
fig_3d = plt.figure(figsize=(18, 14), constrained_layout=True)
axes = [fig_3d.add_subplot(2, 2, i+1, projection="3d") for i in range(4)]

plot_height_surface(
    axes[0], fourier_height_map, tomo_az, tomo_rg,
    "Fourier — Peak scattering height",
)
plot_height_surface(
    axes[1], capon_height_map, tomo_az, tomo_rg,
    "Capon — Peak scattering height",
)
plot_height_surface(
    axes[2], music_height_map, tomo_az, tomo_rg,
    f"{MUSIC_LABEL} — Peak scattering height",
)
plot_height_surface(
    axes[3], gedi_height, gedi_az, gedi_rg,
    "GEDI — Canopy height",
)
add_height_colorbar(fig_3d, axes)
fig_3d.suptitle(
    "BIOMASS TomoSAR — Fourier, Capon, MUSIC and GEDI",
    fontsize=16,
)

save_plot_set("comparison_3d", {"fourier_capon_music_gedi_3d": fig_3d})
plt.show()

---
### Notes on interpretation

- **Fourier / Capon:** reconstructed radar power. **MUSIC:** pseudo-spectrum, not a calibrated scattering power estimate; the panels intentionally use different color limits when not normalized.
- **GEDI:** `rh_98 - rh_0` canopy height; it need not coincide with the strongest P-band XP scattering height.
- **GEDI-based filter:** applied only to the 2D/3D peak-height maps. The original profiles and cut tomograms remain untouched.
- **Output organization:** every plotting cell saves its outputs in a unique folder under the configured `outputs` directory. Every range, azimuth, and pixel folder contains *two files*, one SLC overview and one tomogram/profile figure.